# Block 1 — Document normalization (live `src/med_doc`)

Warp, section layout, crop-window gate. **Does not classify ticks.**

Open in Colab from branch **`block1`**. Do **not** upload clinic PHI — this notebook uses the committed blank form.


## 0. Clone the live tree


In [ ]:
# Colab does not clone src/ when you open a GitHub notebook. Pull the live tree.
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/RwaRwa599/epq3.git"
BRANCH = "block1"

def _run(cmd):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.check_call(cmd)

def ensure_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, here.parent, Path("/content/epq3")]:
        if (cand / "src" / "med_doc").is_dir() and (cand / "pyproject.toml").exists():
            os.chdir(cand)
            return cand
    dest = Path("/content/epq3") if Path("/content").is_dir() else (here / "epq3")
    if not (dest / ".git").is_dir():
        url = REPO
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        if token:
            url = f"https://{token}@github.com/RwaRwa599/epq3.git"
        _run(["git", "clone", "--branch", BRANCH, "--single-branch", url, str(dest)])
    else:
        _run(["git", "-C", str(dest), "fetch", "origin", BRANCH])
        _run(["git", "-C", str(dest), "checkout", BRANCH])
        _run(["git", "-C", str(dest), "pull", "--ff-only", "origin", BRANCH])
    os.chdir(dest)
    return dest

root = ensure_repo()
_run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "matplotlib"])
print("cwd:", os.getcwd())
print("HEAD:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 1. Helpers


In [ ]:
from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/pipeline") if Path("/content").is_dir() else Path("outputs/colab_pipeline")
OUT.mkdir(parents=True, exist_ok=True)

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_sheet() -> Path:
    from med_doc.paths import SYNTHETIC_DIR
    sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
    assert sheet.exists(), sheet
    return sheet

## 2. Normalize the synthetic blank

`normalize_document` returns a canonical canvas + checkbox/handwriting crops. `normalize_batch` writes `block1_normalized_batch.zip` for Blocks 3–5.


In [ ]:
from med_doc.normalization import normalize_document
from med_doc.normalization.batch import normalize_batch

sheet = demo_sheet()
print("input:", sheet)

result = normalize_document(str(sheet), document_id=sheet.stem)
print("template:", result.extra.get("template_id"))
print("warp:", result.warp_method, "orientation:", result.orientation_degrees)
print("alignment:", round(float(result.alignment_confidence), 3))
print("checkboxes:", len(result.checkbox_crops), "handwriting:", len(result.handwriting_crops))

b1 = normalize_batch([sheet], output_dir=OUT / "b1", output_zip=OUT / "block1.zip")
print("ZIP:", b1["output_zip"], "ok", b1["manifest"]["successful_documents"])

## 3. Canonical canvas and overlay


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(result.canonical_canvas)
axes[0].set_title(f"canonical {result.canonical_canvas.shape[1]}x{result.canonical_canvas.shape[0]}")
axes[0].axis("off")
if result.debug_overlay is not None:
    axes[1].imshow(result.debug_overlay)
    axes[1].set_title("debug overlay")
else:
    axes[1].axis("off")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. Crop gallery


In [ ]:
cb_items = list(result.checkbox_crops.items())[:12]
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, (fid, crop) in zip(axes.ravel(), cb_items):
    ax.imshow(crop.normalized_image)
    ax.set_title(f"{fid}\nq={crop.quality_score:.2f}", fontsize=8)
    ax.axis("off")
plt.suptitle("Checkbox crops (Block 1 does not label ticks)")
plt.tight_layout()
plt.show()

hw_items = list(result.handwriting_crops.items())[:6]
fig, axes = plt.subplots(len(hw_items), 1, figsize=(10, 2.2 * max(len(hw_items), 1)))
if len(hw_items) == 1:
    axes = [axes]
for ax, (fid, crop) in zip(axes, hw_items):
    ax.imshow(crop.normalized_image)
    ax.set_title(fid)
    ax.axis("off")
plt.suptitle("Handwriting ROIs")
plt.tight_layout()
plt.show()

## 5. Download Block 1 ZIP


In [ ]:
download(OUT / 'block1.zip')